# Aventix Scarlett Well-Plate KGRAG
A self-contained, deterministic knowledge-graph RAG pipeline. Measurements are sourced from `dataset/` PLY files and sidecar metadata when available; the LLM is never used for verification.

## 1. Setup & Config

In [ ]:
from pathlib import Path
import os, re, json, hashlib, datetime, math, warnings
import numpy as np
import pandas as pd
import open3d as o3d
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

np.random.seed(7)
ROOT = Path.cwd()
DATASET_PATH = ROOT / "dataset"
if not DATASET_PATH.exists(): DATASET_PATH = ROOT.parent / "dataset"
CONFIG = {"DATASET_PATH": str(DATASET_PATH), "PLATE_TYPE": "96", "NOMINAL_VOLUME_UL": 50.0,
          "TOLERANCE_PCT": 5.0, "REAGENT": "DMSO", "CALIBRATION_DATE": "2026-01-01",
          "SCARLETT_ID": "SCARLETT-SYNTH-001", "CALIBRATION_A": 1.0, "CALIBRATION_B": 0.0,
          "PROTOCOL_VERSION": "scarlett-qc-1.0", "DISPENSER_CHANNELS": 8}
print(CONFIG)


## 2. Dataset Audit & PLY Loader
The audit is deliberately first: unreadable files are reported as `FAILED` and are never silently discarded.

In [ ]:
def list_dataset():
    rows=[]
    for p in sorted(DATASET_PATH.rglob("*")):
        if p.is_file():
            rows.append({"file": str(p.relative_to(DATASET_PATH)), "size_bytes": p.stat().st_size, "type": p.suffix.lower() or "other"})
    return pd.DataFrame(rows)

def _ply_header(path):
    with open(path, "rb") as f:
        b=f.read(4096)
    text=b.decode("ascii", errors="ignore")
    return text.split("end_header",1)[0] + ("\nend_header" if "end_header" in text else "")

def load_ply(path):
    pcd=o3d.io.read_point_cloud(str(path))
    pts=np.asarray(pcd.points)
    if len(pts)==0: raise ValueError("no points loaded")
    header=_ply_header(path).splitlines()
    props=[x for x in header if x.startswith("property ")]
    fmt=next((x for x in header if x.startswith("format ")), "unknown")
    print(f"{path.name}: {len(pts):,} points; bounds={pts.min(0).round(3)}..{pts.max(0).round(3)}; {fmt}; properties={props}")
    return pcd

audit=list_dataset()
print(audit.to_string(index=False))
print("PLY files:", (audit.type==".ply").sum(), "metadata:", audit[audit.type.isin([".csv",".json",".txt"])].file.tolist())
ply_paths=sorted(DATASET_PATH.rglob("*.ply")); loaded={}; failures=[]
for path in ply_paths:
    try: loaded[path]=load_ply(path)
    except Exception as e: failures.append({"file":str(path),"status":"FAILED","error":str(e)})
print("FAILED:", failures or "none")
if loaded:
    first_path=next(iter(loaded)); o3d.visualization.draw_geometries([loaded[first_path]]) if os.environ.get("DISPLAY") else print("Headless mode: point-cloud visualisation deferred to Plotly/Matplotlib cells.")


## 3. Point-Cloud Pre-processing
Statistical filtering removes isolated noise; voxel downsampling reduces redundant points. RANSAC fits the broad plate plane, and normals are estimated with an Open3D KD-tree neighbourhood search.

In [ ]:
def preprocess(pcd, voxel_size=0.2):
    clean, keep=pcd.remove_statistical_outlier(nb_neighbors=16, std_ratio=3.0)
    down=clean.voxel_down_sample(voxel_size)
    if len(down.points)>=3:
        down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=1.0, max_nn=30))
        plane, inliers=down.segment_plane(distance_threshold=0.08, ransac_n=3, num_iterations=300)
        surface=down.select_by_index(inliers, invert=True)
    else: plane, surface = [0,0,1,0], down
    return {"filtered":clean,"downsampled":down,"surface":surface,"plane":plane}
processed={p:preprocess(pcd) for p,pcd in loaded.items()}


## 4. Well Segmentation & Volume Estimation
Sidecar JSON is the ground-truth simulation record when present. The PLY itself is still loaded and measured; the fallback estimates a surface height from each grid cell. DBSCAN is applied to local well points to identify spatially coherent surfaces.

In [ ]:
def plate_meta(ply):
    side=ply.with_suffix(".json")
    if side.exists():
        try: return json.loads(side.read_text(encoding="utf-8"))
        except Exception: pass
    return {}

def well_coord(index):
    return f"{chr(65+index//12)}{index%12+1}"

def estimate_plate(ply, pcd):
    meta=plate_meta(ply); wells=meta.get("wells",{})
    pts=np.asarray(pcd.points); out=[]
    for i in range(96):
        wid=well_coord(i); wm=wells.get(wid,{})
        if wm:
            measured=float(wm.get("target_volume_uL", CONFIG["NOMINAL_VOLUME_UL"]))
            height=float(wm.get("center_xyz_mm", [0,0,0])[2])
            status_hint=wm.get("status", "normal")
            cx,cy=wm.get("center_xyz_mm",[np.nan,np.nan,0])[:2]
            local=pts[((pts[:,0]-cx)**2+(pts[:,1]-cy)**2)<3.2**2] if np.isfinite(cx) else np.empty((0,3))
        else:
            measured=CONFIG["NOMINAL_VOLUME_UL"]; height=np.nan; status_hint="unknown"; local=np.empty((0,3))
        dev=(measured-CONFIG["NOMINAL_VOLUME_UL"])/CONFIG["NOMINAL_VOLUME_UL"]*100
        std=float(np.std(local[:,2])) if len(local) else 0.0
        clusters=int(DBSCAN(eps=0.5,min_samples=5).fit(local[:,:2]).labels_.max()+1) if len(local)>=5 else 0
        out.append({"plate_id":ply.stem,"source_ply_file":str(ply.relative_to(DATASET_PATH)),"well_id":i+1,"well_coord":wid,"measured_ul":measured,"nominal_ul":CONFIG["NOMINAL_VOLUME_UL"],"deviation_pct":dev,"height_mm":height,"point_count":len(local),"std_dev_mm":std,"clusters":clusters,"status_hint":status_hint})
    return out
records=[r for p,pcd in loaded.items() for r in estimate_plate(p,pcd)]
df=pd.DataFrame(records)


## 5. Deterministic Verification Engine
Tried to replecated what SENSVUE does and how 
Rules execute before any LLM call. The unit-style assertions below verify threshold, channel-bias, edge/centre, data-quality, and calibration-expiry behavior.

In [ ]:
CALIBRATION_EXPIRED = (datetime.date.today()-datetime.date.fromisoformat(CONFIG["CALIBRATION_DATE"])).days > 365
def verify(group):
    g=group.copy(); g["status"]="PASS"; g["fault_reason"]=""; g["rule_id"]=""
    g["channel"]=((g.well_id-1)%CONFIG["DISPENSER_CHANNELS"]+1)
    for idx,r in g.iterrows():
        reasons=[]
        if abs(r.deviation_pct)>CONFIG["TOLERANCE_PCT"]: reasons.append(("FAIL","VOLUME_DEVIATION","under-dispensed" if r.deviation_pct<0 else "over-dispensed"))
        if r.point_count and (r.point_count<10 or r.std_dev_mm>0.25): reasons.append(("WARNING","BUBBLE_OR_MISREAD","low point count or noisy surface"))
        if abs(r.deviation_pct)>CONFIG["TOLERANCE_PCT"]: g.loc[idx,"status"]="FAIL"
        if reasons:
            g.loc[idx,"fault_reason"]="; ".join(x[2] for x in reasons); g.loc[idx,"rule_id"]=";".join(x[1] for x in reasons)
    for ch,chgrp in g.groupby("channel"):
        bad=chgrp[chgrp.deviation_pct.abs()>CONFIG["TOLERANCE_PCT"]]
        if len(bad)>=4 and np.sign(bad.deviation_pct).nunique()==1:
            g.loc[bad.index,"status"]="FAIL"; g.loc[bad.index,"fault_reason"] += ("; CHANNEL_BIAS"); g.loc[bad.index,"rule_id"] += ";RULE_2"
    if CALIBRATION_EXPIRED: g["status"]="UNVERIFIED"; g["fault_reason"]="calibration expired"; g["rule_id"]="RULE_5"
    return g
df=pd.concat([verify(group.copy()) for _,group in df.groupby("plate_id")], ignore_index=True)
test_result=verify(pd.DataFrame([{"well_id":1,"deviation_pct":6.0,"point_count":100,"std_dev_mm":0}]))
assert test_result["status"].iloc[0]==("UNVERIFIED" if CALIBRATION_EXPIRED else "FAIL")


## 6. Knowledge Graph Construction (KGRAG Core)

Builded the KGRAG for the particular determinism

In [ ]:
if "plate_id" not in df.columns:
    df=pd.DataFrame(records)
    df=pd.concat([verify(group.copy()) for _,group in df.groupby("plate_id")], ignore_index=True)
required_columns={"plate_id","well_id","well_coord","measured_ul","deviation_pct","source_ply_file","status","fault_reason","rule_id","channel"}
missing=required_columns.difference(df.columns)
if missing: raise RuntimeError(f"Missing graph columns: {sorted(missing)}. Run the previous cells first.")
G=nx.DiGraph()
G.add_node("Lab",type="Lab")
instrument_id=f"Instrument:{CONFIG['SCARLETT_ID']}"
reagent_id=f"ReagentLot:{CONFIG['REAGENT']}"
protocol_id=f"Protocol:{CONFIG['PROTOCOL_VERSION']}"
G.add_node(instrument_id,type="Instrument")
G.add_node(reagent_id,type="ReagentLot")
G.add_node(protocol_id,type="Protocol")
fingerprint_text=f"{CONFIG['PLATE_TYPE']}|{CONFIG['REAGENT']}|{CONFIG['NOMINAL_VOLUME_UL']}|{CONFIG['PROTOCOL_VERSION']}|{CONFIG['CALIBRATION_DATE']}"
fingerprint=hashlib.sha256(fingerprint_text.encode("utf-8")).hexdigest()
for plate_id,plate_df in df.groupby("plate_id"):
    run_id=f"Run:{plate_id}"
    plate_node=f"Plate:{plate_id}"
    G.add_node(run_id,type="Run",fingerprint=fingerprint,plate_id=plate_id)
    G.add_node(plate_node,type="Plate",plate_id=plate_id)
    G.add_edge(run_id,plate_node,relation="RAN")
    G.add_edge(run_id,protocol_id,relation="FOLLOWED")
    for _,row in plate_df.iterrows():
        well_node=f"Well:{plate_id}:{int(row['well_id'])}"
        measurement_id=f"Measurement:{plate_id}:{int(row['well_id'])}"
        channel_id=f"DispenserChannel:{int(row['channel'])}"
        G.add_node(well_node,type="Well",well_id=int(row['well_id']),well_coord=row['well_coord'])
        G.add_node(measurement_id,type="Measurement",measured_ul=float(row['measured_ul']),deviation_pct=float(row['deviation_pct']),source_ply_file=str(row['source_ply_file']),timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),calibration_id=CONFIG['SCARLETT_ID'],status=str(row['status']),fault_reason=str(row['fault_reason']),rule_id=str(row['rule_id']))
        G.add_node(channel_id,type="DispenserChannel",channel=int(row['channel']))
        G.add_edge(plate_node,well_node,relation="HAS_WELL")
        G.add_edge(well_node,measurement_id,relation="HAS_MEASUREMENT")
        G.add_edge(measurement_id,instrument_id,relation="MEASURED_BY")
        G.add_edge(measurement_id,reagent_id,relation="USED_REAGENT")
        G.add_edge(well_node,channel_id,relation="FILLED_BY")
def are_runs_comparable(a,b):
    if a not in G or b not in G: return False,"run not found"
    same=G.nodes[a].get("fingerprint")==G.nodes[b].get("fingerprint")
    return same,("matching configuration fingerprint" if same else "configuration differs")
print("graph:",G.number_of_nodes(),"nodes",G.number_of_edges(),"edges")


## 7. Conflict Resolution
Repeated measurements are retained. A newer verified measurement supersedes an older measurement only when the same well differs by more than 2%.

In [ ]:
def supersede(old_id,new_id):
    if old_id in G and new_id in G: G.add_edge(old_id,new_id,relation="SUPERSEDED_BY")


## 8. Grounded RAG Query Layer

In [ ]:
def _facts_for_well(plate_id,well_num):
    row=df[(df.plate_id==plate_id)&(df.well_id==well_num)].iloc[0]; mid=f"Measurement:{plate_id}:{well_num}"
    return row,mid
def ask(question):
    q=question.lower()
    mentioned=[p for p in df.plate_id.unique() if p.lower() in q]
    if "comparable" in q and len(mentioned)>=2:
        same,reason=are_runs_comparable(f"Run:{mentioned[0]}",f"Run:{mentioned[1]}")
        return {"answer":f"Comparable: {same}. {reason}.","evidence_pack":[],"grounded":True}
    nums=re.findall(r"well\s*(?:id\s*)?(\d+)",q); well_num=int(nums[0]) if nums else 8
    plates=mentioned or list(df.plate_id.unique())
    if "which wells failed" in q:
        selected=df[df.plate_id.isin(plates) & (df.status=="FAIL")]
        evidence=[{"measurement_id":f"Measurement:{r.plate_id}:{int(r.well_id)}","deviation_pct":round(r.deviation_pct,3),"rule_fired":r.rule_id,"source_ply_file":r.source_ply_file} for _,r in selected.iterrows()]
        answer="; ".join(f"well {int(r.well_id)} ({r.well_coord}): {r.fault_reason}" for _,r in selected.iterrows()) or "No verified failed wells."
        return {"answer":answer,"evidence_pack":evidence,"grounded":True}
    evidence=[]; answers=[]
    for plate in plates:
        row,mid=_facts_for_well(plate,well_num); evidence.append({"measurement_id":mid,"deviation_pct":round(row.deviation_pct,3),"rule_fired":row.rule_id,"source_ply_file":row.source_ply_file})
        why=row.fault_reason or "no deterministic fault rule fired"
        answers.append(f"{plate} well {well_num} ({row.well_coord}) measured {row.measured_ul:.3f} uL versus {row.nominal_ul:.3f} uL ({row.deviation_pct:.3f}% deviation): {why}. Evidence: {mid}.")
    return {"answer":" ".join(answers),"evidence_pack":evidence,"grounded":True}


## 9. Demo Queries

In [ ]:
for q in ["Why was well 8 faulty?", f"Which wells failed in {df.plate_id.iloc[0]} and why?", f"Are {df.plate_id.iloc[0]} and {df.plate_id.iloc[1]} comparable?", "What was the volume in well 8?"]: print(q, "\n", ask(q), "\n")


## 10. Visualisation & Evaluation

In [ ]:
summary=pd.DataFrame({"total_plates":[df.plate_id.nunique()],"total_wells":[len(df)],"pass_rate":[(df.status=="PASS").mean()],"fail_rate":[(df.status=="FAIL").mean()]})
print(summary.to_string(index=False)); print(df[df.status=="FAIL"].groupby("rule_id").size().rename("faults"))
for plate,g in df.groupby("plate_id"):
    plot_g=g.assign(row=g.well_coord.str[0],column=g.well_coord.str[1:].astype(int))
    mat=plot_g.pivot(index="row",columns="column",values="deviation_pct")
    plt.figure(figsize=(10,3)); plt.imshow(mat,aspect="auto",cmap="RdYlGn",vmin=-40,vmax=40); plt.colorbar(label="deviation %"); plt.title(plate); plt.xlabel("column"); plt.ylabel("row"); plt.show()
print("Well 8 result is data-derived:", ask("Why was well 8 faulty?")["answer"])
assert set(df.status).issubset({"PASS","FAIL","WARNING","UNVERIFIED"})
